In [1]:
# testing hysteresis parameter calculation? 
import numpy as np
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import matplotlib.dates as mdates

shear_stress = pd.read_csv('data/shear_stress/average_total_shear_stress_corrected.csv', parse_dates=['datetime'], index_col='datetime')
sonde_downstream = pd.read_csv('data/sonde_data/sonde_down_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')
sonde_upstream = pd.read_csv('data/sonde_data/sonde_up_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')

Process and Merge Data

In [2]:
# average rows with duplicate timestamps on sonde data (second-resolution data)
sonde_downstream = sonde_downstream.groupby(level=0).mean(numeric_only=True)
sonde_upstream = sonde_upstream.groupby(level=0).mean(numeric_only=True)

# re-sample shear stress and sonde data to 1 min intervals
shear_stress = shear_stress.resample('1min').interpolate()
sonde_downstream_resampled = sonde_downstream.resample('1min').interpolate()
sonde_upstream_resampled = sonde_upstream.resample('1min').interpolate()

# join the sonde data with the shear stress data for the entire record 
sonde_downstream = sonde_downstream.join(shear_stress, how="left")
sonde_upstream = sonde_upstream.join(shear_stress, how="left")

Hysteresis index calculation functions

In [3]:
def normalize_storm_df(df, tau_col, constituent_cols, cols_to_normalize=None):
    out = df.copy()

    if cols_to_normalize is None:
        cols_to_normalize = [tau_col] + [c for c in constituent_cols if c in out.columns]

    for col in cols_to_normalize:
        if col not in out.columns:
            continue

        series = out[col].astype(float)
        smin = series.min(skipna=True)
        smax = series.max(skipna=True)

        if pd.isna(smin) or pd.isna(smax) or smax == smin:
            out[col] = np.nan
        else:
            out[col] = (series - smin) / (smax - smin)

    return out

def split_hydrograph(df, tau_col, constituent_col):
    ts = df[[tau_col, constituent_col]].dropna().sort_index()
    if ts.empty:
        return None, None, None

    peak_pos = ts[tau_col].to_numpy().argmax()
    peak_label = ts.index[peak_pos]

    rise = ts.iloc[:peak_pos + 1][[tau_col, constituent_col]]
    fall = ts.iloc[peak_pos:][[tau_col, constituent_col]]  # exclude the peak point from the falling limb
    return rise, fall, peak_label

def interpolate_time_at_tau(tau_target, limb_df, tau_col, allow_extrapolate=True, fit_points=4):
    if limb_df is None or limb_df.empty:
        return None

    limb = limb_df[[tau_col]].dropna().sort_index()
    if len(limb) == 0:
        return None
    if len(limb) == 1:
        return limb.index[0]

    tau = limb[tau_col].to_numpy(dtype=float)
    t = limb.index.view("int64")

    exact = np.where(tau == tau_target)[0]
    if exact.size > 0:
        return limb.index[exact[0]]

    diffs = tau - tau_target
    crossings = np.where(diffs[:-1] * diffs[1:] <= 0)[0]

    if crossings.size > 0:
        i = crossings[0]
        tau0, tau1 = tau[i], tau[i + 1]
        t0, t1 = t[i], t[i + 1]
    else:
        if not allow_extrapolate:
            return None

        n = min(fit_points, len(tau))
        if n < 2:
            return limb.index[-1]

        if tau_target < np.nanmin(tau):
            tau_fit = tau[:n]
            t_fit = t[:n]
        else:
            tau_fit = tau[-n:]
            t_fit = t[-n:]

        if np.all(tau_fit == tau_fit[0]):
            return pd.to_datetime(int(t_fit[0]))

        slope, intercept = np.polyfit(tau_fit, t_fit, 1)
        t_target = slope * tau_target + intercept
        return pd.to_datetime(int(t_target))

    if tau1 == tau0:
        return pd.to_datetime(int(t0))

    t_target = t0 + (t1 - t0) * ((tau_target - tau0) / (tau1 - tau0))
    return pd.to_datetime(int(t_target))

def get_time_bracket_points(t_target, limb_df, tau_col, constituent_col, allow_extrapolate=True, fit_points=4):
    if limb_df is None or limb_df.empty:
        return None

    limb = limb_df[[tau_col, constituent_col]].dropna().sort_index()
    if len(limb) < 2:
        return None

    t = limb.index.view("int64")
    tau = limb[tau_col].to_numpy(dtype=float)
    c = limb[constituent_col].to_numpy(dtype=float)
    t_target_int = pd.Timestamp(t_target).value

    extrapolated = False

    if t[0] <= t_target_int <= t[-1]:
        upper_idx = np.searchsorted(t, t_target_int, side="right")
        lower_idx = max(upper_idx - 1, 0)
        upper_idx = min(upper_idx, len(t) - 1)
    elif not allow_extrapolate:
        return None
    else:
        n = min(fit_points, len(t))
        if n < 2:
            return None

        extrapolated = True
        if t_target_int < t[0]:
            lower_idx, upper_idx = 0, n - 1
        else:
            lower_idx, upper_idx = len(t) - n, len(t) - 1

    return {
        "t0": pd.to_datetime(int(t[lower_idx])),
        "t1": pd.to_datetime(int(t[upper_idx])),
        "tau0": tau[lower_idx],
        "tau1": tau[upper_idx],
        "c0": c[lower_idx],
        "c1": c[upper_idx],
        "extrapolated": extrapolated,
    }

def interpolate_constituent_at_time(t_target, limb_df, constituent_col, allow_extrapolate=True):
    if limb_df is None or limb_df.empty:
        return np.nan

    limb = limb_df[[constituent_col]].dropna().sort_index()
    if len(limb) == 0:
        return np.nan
    if len(limb) == 1:
        return limb[constituent_col].iloc[0]

    t = limb.index.view("int64")
    y = limb[constituent_col].to_numpy(dtype=float)
    t_target_int = pd.Timestamp(t_target).value

    if t[0] <= t_target_int <= t[-1]:
        return np.interp(t_target_int, t, y)

    if not allow_extrapolate:
        return np.nan

    if t_target_int < t[0]:
        t0, t1 = t[0], t[1]
        y0, y1 = y[0], y[1]
    else:
        t0, t1 = t[-2], t[-1]
        y0, y1 = y[-2], y[-1]

    if t1 == t0:
        return y0

    slope = (y1 - y0) / (t1 - t0)
    return y0 + slope * (t_target_int - t0)

def get_time_bracket_points(t_target, limb_df, tau_col, constituent_col, allow_extrapolate=True, fit_points=4):
    if limb_df is None or limb_df.empty:
        return None

    limb = limb_df[[tau_col, constituent_col]].dropna().sort_index()
    if len(limb) < 2:
        return None

    t = limb.index.view("int64")
    tau = limb[tau_col].to_numpy(dtype=float)
    c = limb[constituent_col].to_numpy(dtype=float)
    t_target_int = pd.Timestamp(t_target).value

    extrapolated = False

    if t[0] <= t_target_int <= t[-1]:
        upper_idx = np.searchsorted(t, t_target_int, side="right")
        lower_idx = max(upper_idx - 1, 0)
        upper_idx = min(upper_idx, len(t) - 1)
    elif not allow_extrapolate:
        return None
    elif t_target_int < t[0]:
        lower_idx, upper_idx = 0, 1
        extrapolated = True
    else:
        lower_idx, upper_idx = len(t) - 2, len(t) - 1
        extrapolated = True

    return {
        "t0": pd.to_datetime(int(t[lower_idx])),
        "t1": pd.to_datetime(int(t[upper_idx])),
        "tau0": tau[lower_idx],
        "tau1": tau[upper_idx],
        "c0": c[lower_idx],
        "c1": c[upper_idx],
        "extrapolated": extrapolated,
    }

def calculate_lawler_adapted_HI(df, tau_col, constituent_col, storm_name, k=0.5):
    rise, fall, peak_label = split_hydrograph(df, tau_col, constituent_col)
    if rise is None or fall is None or rise.empty or fall.empty:
        print(f"Not enough data points to split the hydrograph for {storm_name} {constituent_col}")
        return None

    tau_min = df[tau_col].min()
    tau_max = df[tau_col].max()
    tau_mid = k * (tau_max - tau_min) + tau_min

    t_rise_mid = interpolate_time_at_tau(tau_mid, rise, tau_col, allow_extrapolate=True)
    t_fall_mid = interpolate_time_at_tau(tau_mid, fall, tau_col, allow_extrapolate=True)

    if t_rise_mid is None or t_fall_mid is None:
        print(f"Could not find target time for storm {storm_name} {constituent_col}")
        return None

    c_rise = interpolate_constituent_at_time(t_rise_mid, rise, constituent_col, allow_extrapolate=True)
    c_fall = interpolate_constituent_at_time(t_fall_mid, fall, constituent_col, allow_extrapolate=True)

    if np.isnan(c_rise) or np.isnan(c_fall):
        print(f"Could not interpolate storm {storm_name} {constituent_col}")
        return None

    rise_bracket = get_time_bracket_points(t_rise_mid, rise, tau_col, constituent_col)
    fall_bracket = get_time_bracket_points(t_fall_mid, fall, tau_col, constituent_col)

    if c_rise > c_fall:
        HI = 1 - (c_fall / c_rise)
    elif c_rise < c_fall:
        HI = -1 + (c_rise / c_fall)
    else:
        HI = 0

    return {
        "storm_name": storm_name,
        "HI": HI,
        "tau_mid": tau_mid,
        "c_rise": c_rise,
        "c_fall": c_fall,
        "rise": rise,
        "fall": fall,
        "tau_col": tau_col,
        "constituent_col": constituent_col,
        "k": k,
        "t_rise_mid": t_rise_mid,
        "t_fall_mid": t_fall_mid,
        "rise_bracket": rise_bracket,
        "fall_bracket": fall_bracket,
        "peak_label": peak_label,
    }

def calculate_lloyd_HI(df, tau_col, constituent_col, storm_name, k=0.5):
    rise, fall, peak_label = split_hydrograph(df, tau_col, constituent_col)
    if rise is None or fall is None or rise.empty or fall.empty:
        print(f"Not enough data points to split the hydrograph for {storm_name} {constituent_col}")
        return None

    tau_min = df[tau_col].min()
    tau_max = df[tau_col].max()
    tau_mid = k * (tau_max - tau_min) + tau_min

    t_rise_mid = interpolate_time_at_tau(tau_mid, rise, tau_col, allow_extrapolate=True)
    t_fall_mid = interpolate_time_at_tau(tau_mid, fall, tau_col, allow_extrapolate=True)

    if t_rise_mid is None or t_fall_mid is None:
        print(f"Could not find target time for storm {storm_name} {constituent_col}")
        return None

    c_rise = interpolate_constituent_at_time(t_rise_mid, rise, constituent_col, allow_extrapolate=True)
    c_fall = interpolate_constituent_at_time(t_fall_mid, fall, constituent_col, allow_extrapolate=True)

    if np.isnan(c_rise) or np.isnan(c_fall):
        print(f"Could not interpolate storm {storm_name} {constituent_col}")
        return None

    rise_bracket = get_time_bracket_points(t_rise_mid, rise, tau_col, constituent_col)
    fall_bracket = get_time_bracket_points(t_fall_mid, fall, tau_col, constituent_col)

    HI = c_rise - c_fall

    return {
        "storm_name": storm_name,
        "HI": HI,
        "tau_mid": tau_mid,
        "c_rise": c_rise,
        "c_fall": c_fall,
        "rise": rise,
        "fall": fall,
        "tau_col": tau_col,
        "constituent_col": constituent_col,
        "k": k,
        "t_rise_mid": t_rise_mid,
        "t_fall_mid": t_fall_mid,
        "rise_bracket": rise_bracket,
        "fall_bracket": fall_bracket,
        "peak_label": peak_label,
    }

def plot_lloyd_hysteresis_multi_k(results_list, df, tau_col, constituent_col,
                                    xlabel=r"Shear Stress ($\tau$)",
                                    ylabel="Constituent",
                                    out_dir="HI_calculations/lawler_plots",
                                    save=True,
                                    show=False):

    if results_list is None or len(results_list) == 0:
        return

    results_list = sorted(results_list, key=lambda r: r.get("k", 0.5))

    storm_name = results_list[0].get("storm_name", "")
    ts_df = df[[tau_col, constituent_col]].dropna().sort_index()

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={"width_ratios": [1.2, 1]})

    # Colors for the different k values
    cmap = plt.get_cmap("viridis")
    color_values = np.linspace(0.15, 0.85, len(results_list))

    # Left panel: time series with horizontal threshold lines for each k
    ax = axes[0]
    ax.plot(ts_df.index, ts_df[tau_col], color="tab:blue", linewidth=1.5, label=tau_col)
    ax.set_ylabel(tau_col, color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.xaxis.set_major_locator(MaxNLocator(8))

    ax2 = ax.twinx()
    ax2.plot(ts_df.index, ts_df[constituent_col], color="tab:red", linewidth=1.5, label=constituent_col)
    ax2.set_ylabel(constituent_col, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    for result in results_list:
        tau_mid = result["tau_mid"]
        k = result.get("k", 0.5)
        ax.axhline(
            tau_mid,
            linestyle="--",
            color="gray",
            alpha=0.7,
            linewidth=1.3,
            label=fr"$\tau_{{mid}}$ (k={k:.1f})"
        )

    peak_label = results_list[0].get("peak_label")
    if peak_label is not None and peak_label in ts_df.index:
        tau_peak = ts_df.at[peak_label, tau_col]
        ax.scatter(peak_label, tau_peak, marker="*", color="k", s=160, zorder=9)

    event_date = ts_df.index[0].strftime("%Y-%m-%d")
    ax.set_title(f"Event time series ({event_date})")
    ax.set_xlabel("Time (HH:MM)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.grid(True, alpha=0.3)

    # Right panel: hysteresis plot with one gray vertical line per k
    ax = axes[1]
    point_size = 35

    rise = results_list[0]["rise"]
    fall = results_list[0]["fall"]

    combined = pd.concat([rise, fall.iloc[1:]])
    ax.plot(combined[tau_col], combined[constituent_col], color="black", linewidth=2, alpha=0.6, zorder=1)

    # Base points: no black edges here
    ax.scatter(rise[tau_col], rise[constituent_col], color="gray", s=point_size,
                label="Rising limb", zorder=4, edgecolor="none")
    ax.scatter(fall[tau_col], fall[constituent_col], color="gray", s=point_size,
                label="Falling limb", zorder=4, edgecolor="none")

    for result in results_list:
        tau_mid = result["tau_mid"]
        c_rise = result["c_rise"]
        c_fall = result["c_fall"]
        k = result.get("k", 0.5)

        ax.axvline(
            tau_mid,
            linestyle="--",
            color="gray",
            linewidth=1.5,
            alpha=0.7,
            label=fr"$\tau_{{mid}}$ (k={k:.1f})"
        )

        # Only the tau_target points get black edges
        ax.scatter(tau_mid, c_rise, s=point_size, color="forestgreen",
                    edgecolor="black", linewidth=1.2, zorder=6)
        ax.scatter(tau_mid, c_fall, s=point_size, color="firebrick",
                    edgecolor="black", linewidth=1.2, zorder=6)

    if peak_label is not None and peak_label in ts_df.index:
        tau_peak = ts_df.at[peak_label, tau_col]
        c_peak = ts_df.at[peak_label, constituent_col]
        ax.scatter(tau_peak, c_peak, marker="*", color="k", s=160, zorder=9, edgecolor="black")

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_title("Lawler HI across k values")

    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    fig.suptitle(main_title, fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = f"{storm_name}_{constituent_col}_lawler_multi_k.png".replace(" ", "_").replace("/", "_")
        fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()

    plt.close(fig)

Separating storm by date and time

In [4]:
# define storm windows once
storm_windows = {
    "st1_down": ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st1_up":   ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st2_down": ("2022-08-03 15:00", "2022-08-04 2:00"),
    "st2_up":   ("2022-08-03 15:00", "2022-08-03 19:00"),
    "st3_down": ("2022-08-08 12:30", "2022-08-09 19:00"),
    "st3_up":   ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st4_down": ("2023-07-29 12:30", "2023-07-30 5:30"),
    "st4_up":   ("2023-07-29 13:00", "2023-07-30 19:30"),
    "st5_down": ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st5_up":   ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st6_down": ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st6_up":   ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st7_down": ("2023-09-14 13:00", "2023-09-15 13:00"),
    "st7_up":   ("2023-09-14 13:00", "2023-09-15 13:00")
}

# build event dictionary directly (no function)
sonde_events = {}
down = sonde_downstream.sort_index()   
up = sonde_upstream.sort_index()      

for storm_name, (start, end) in storm_windows.items():
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    if "down" in storm_name.lower():
        src = down
    elif "up" in storm_name.lower():
        src = up
    else:
        continue
    sonde_events[storm_name] = src.loc[(src.index >= start) & (src.index <= end)].copy()

Normalize shear stress and sonde data

In [5]:
sonde_constituents = ["Turbidity FNU", "fDOM RFU"]

for storm_name, storm_df in sonde_events.items():
    sonde_events[storm_name] = normalize_storm_df(
        storm_df,
        tau_col="shear_stress",
        constituent_cols=sonde_constituents
    )

### Lawler Adapted:
Turbidity and fDOM hysteresis

Calculate Lawler Adapted HI for sonde data

In [6]:
ks = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
sonde_constituents = ["Turbidity FNU", "fDOM RFU"]

for constituent in sonde_constituents:
    summary_rows = []

    for storm_name, storm_df in sonde_events.items():
        if constituent not in storm_df.columns:
            continue

        row = {"storm": storm_name}

        for k in ks:
            result = calculate_lawler_adapted_HI(
                storm_df,
                tau_col="shear_stress",
                constituent_col=constituent,
                storm_name=storm_name,
                k=k
            )

            col_name = f"HI{int(k * 100)}"
            row[col_name] = np.nan if result is None else result["HI"]

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    if not summary_df.empty:
        hi_cols = [f"HI{int(k * 100)}" for k in ks]
        summary_df = summary_df[["storm"] + hi_cols]
        summary_df["HI_avg"] = summary_df[hi_cols].mean(axis=1, skipna=True)
        summary_df = summary_df[["storm"] + hi_cols + ["HI_avg"]]
        summary_df = summary_df.sort_values("storm").reset_index(drop=True)

        out_path = f"HI_calculations/lawler_adapted_{constituent.replace(' ', '_').replace('/', '_').lower()}_hysteresis_summary.csv"
        summary_df.to_csv(out_path, index=False)

        display(summary_df)

Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbid

,storm,HI10,HI20,HI30,HI40,HI50,HI60,HI70,HI80,HI90,HI_avg
0,st1_down,0.472901,-0.704964,-0.327388,0.519563,0.266547,0.314821,0.523868,0.733148,0.635877,0.270486
1,st1_up,0.405917,0.281044,0.321478,0.557689,0.541501,0.611937,0.759808,0.803026,0.716284,0.555409
2,st2_down,0.901970,0.858505,0.793956,0.746005,0.733861,0.667852,0.564929,0.351526,0.407781,0.669598
3,st2_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,st3_down,-0.857933,0.018830,0.457046,0.517496,0.169633,0.142118,0.045465,0.156226,0.267276,0.101795
5,st3_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,st4_down,-10.698990,0.395444,0.491762,0.371692,0.261784,0.306251,0.320523,0.241260,0.134893,-0.908376
7,st4_up,0.943850,0.368086,-0.311039,0.229432,0.529217,0.397178,0.391850,0.286321,0.166292,0.333465
8,st5_down,-0.148863,0.891380,0.797204,0.730011,0.718501,0.689861,0.522118,0.602338,0.618932,0.602387
9,st5_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM

,storm,HI10,HI20,HI30,HI40,HI50,HI60,HI70,HI80,HI90,HI_avg
0,st1_down,-0.322283,-0.679610,-0.806083,-0.779427,-0.570476,-0.508217,-0.647376,-0.729414,-0.565296,-0.623131
1,st1_up,-0.145900,-0.617574,-0.752021,-0.711650,-0.574419,-0.549526,-0.648792,-0.745645,-0.601459,-0.594110
2,st2_down,-0.952678,-0.915582,-0.893280,-0.860823,-0.824558,-0.799717,-0.763756,-0.703897,-0.658319,-0.819179
3,st2_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,st3_down,-0.987082,-0.875002,-0.839979,-0.799112,-0.777639,-0.759336,-0.739169,-0.641540,-0.480724,-0.766620
5,st3_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,st4_down,-1.108507,-0.817954,-0.843656,-0.859288,-0.916411,-0.989072,-0.999442,-0.375182,-0.208878,-0.790932
7,st4_up,-0.487173,-0.754625,-0.624853,-0.578743,-0.644458,-0.627960,-0.614341,-0.248003,-0.134037,-0.523799
8,st5_down,-0.984348,-0.317225,-0.245435,-0.172629,-0.120988,-0.089170,-0.090612,0.181431,0.370154,-0.163203
9,st5_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Plot sonde data

In [7]:
for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        results_list = []

        for k in ks:
            result = calculate_lawler_adapted_HI(
                storm_df,
                tau_col="shear_stress",
                constituent_col=constituent,
                storm_name=storm_name,
                k=k
            )

            if result is not None:
                results_list.append(result)

        if results_list:
            plot_lloyd_hysteresis_multi_k(
                results_list,
                storm_df,
                tau_col="shear_stress",
                constituent_col=constituent,
                out_dir="HI_calculations/lawler_adapted_plots",
                save=True,
                show=False
            )

Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data p

### Lloyd:
Turbidity and fDOM hysteresis

In [8]:
ks = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
sonde_constituents = ["Turbidity FNU", "fDOM RFU"]

for constituent in sonde_constituents:
    summary_rows = []

    for storm_name, storm_df in sonde_events.items():
        if constituent not in storm_df.columns:
            continue

        row = {"storm": storm_name}

        for k in ks:
            result = calculate_lloyd_HI(
                storm_df,
                tau_col="shear_stress",
                constituent_col=constituent,
                storm_name=storm_name,
                k=k
            )

            col_name = f"HI{int(k * 100)}"
            row[col_name] = np.nan if result is None else result["HI"]

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    if not summary_df.empty:
        hi_cols = [f"HI{int(k * 100)}" for k in ks]
        summary_df = summary_df[["storm"] + hi_cols]
        summary_df["HI_avg"] = summary_df[hi_cols].mean(axis=1, skipna=True)
        summary_df = summary_df[["storm"] + hi_cols + ["HI_avg"]]
        summary_df = summary_df.sort_values("storm").reset_index(drop=True)

        out_path = f"HI_calculations/lloyd_{constituent.replace(' ', '_').replace('/', '_').lower()}_hysteresis_summary.csv"
        summary_df.to_csv(out_path, index=False)

        display(summary_df)

Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbidity FNU
Not enough data points to split the hydrograph for st3_up Turbid

,storm,HI10,HI20,HI30,HI40,HI50,HI60,HI70,HI80,HI90,HI_avg
0,st1_down,0.009821,-0.073544,-0.027333,0.071912,0.035096,0.056913,0.199870,0.611188,0.598638,0.164729
1,st1_up,0.010854,0.010775,0.019972,0.081883,0.100498,0.170613,0.453415,0.740972,0.625070,0.246006
2,st2_down,0.623671,0.658159,0.668362,0.666774,0.693322,0.664997,0.555763,0.339597,0.386720,0.584152
3,st2_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,st3_down,-0.072825,0.001085,0.077539,0.141959,0.051727,0.047688,0.016648,0.078866,0.190766,0.059273
5,st3_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,st4_down,-0.656122,0.113398,0.216774,0.194530,0.178422,0.260768,0.306368,0.223477,0.118301,0.106213
7,st4_up,0.086334,0.058500,-0.093569,0.083092,0.338790,0.318386,0.361660,0.246520,0.134030,0.170416
8,st5_down,-0.007810,0.440381,0.450137,0.441303,0.462993,0.472043,0.378081,0.533371,0.570304,0.415645
9,st5_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM RFU
Not enough data points to split the hydrograph for st3_up fDOM

,storm,HI10,HI20,HI30,HI40,HI50,HI60,HI70,HI80,HI90,HI_avg
0,st1_down,-0.097542,-0.407592,-0.769576,-0.775293,-0.535504,-0.442949,-0.535596,-0.588749,-0.400157,-0.505884
1,st1_up,-0.036261,-0.349132,-0.680746,-0.709755,-0.572670,-0.533606,-0.595055,-0.642107,-0.469631,-0.509885
2,st2_down,-0.564010,-0.607712,-0.703505,-0.741759,-0.733893,-0.768139,-0.746938,-0.641686,-0.595015,-0.678073
3,st2_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,st3_down,-0.417397,-0.464178,-0.735450,-0.791236,-0.749082,-0.724053,-0.693646,-0.589996,-0.434220,-0.622140
5,st3_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,st4_down,-0.657546,-0.817827,-0.703497,-0.624142,-0.547803,-0.535335,-0.501271,-0.174546,-0.089315,-0.516809
7,st4_up,-0.063964,-0.498078,-0.624330,-0.440729,-0.481161,-0.404095,-0.369404,-0.142180,-0.072780,-0.344080
8,st5_down,-0.456522,-0.150125,-0.124870,-0.088380,-0.063765,-0.049239,-0.054069,0.133503,0.359674,-0.054866
9,st5_up,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Plot sonde data

In [9]:
for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        results_list = []

        for k in ks:
            result = calculate_lloyd_HI(
                storm_df,
                tau_col="shear_stress",
                constituent_col=constituent,
                storm_name=storm_name,
                k=k
            )

            if result is not None:
                results_list.append(result)

        if results_list:
            plot_lloyd_hysteresis_multi_k(
                results_list,
                storm_df,
                tau_col="shear_stress",
                constituent_col=constituent,
                out_dir="HI_calculations/lloyd_plots",
                save=True,
                show=False
            )

Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up Turbidity FNU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data points to split the hydrograph for st2_up fDOM RFU
Not enough data p